# Deterministic Laplacian coordinates: experiment

Fix both eigenvector signs and rotations within repeated-eigenvalue eigenspaces. Use the eigenspace projector and a fixed sequence of graph-node coordinate axes to construct a canonical orthonormal basis. Complete each repeated-eigenvalue block before taking the first eight coordinates.

This definition assumes the dataset's fixed face ordering; it is not invariant to relabelling nodes. Determinism is checked to numerical tolerance, rather than promised bit-for-bit across numerical libraries. The existing classifier was trained with the legacy basis, so changing its inputs can change predictions. No retraining or published-cache modification is performed.

In [1]:
%matplotlib inline
from pathlib import Path
import sys, json, time
sys.path.insert(0, str(Path.cwd() / "scripts"))
import numpy as np
import torch
from scipy import linalg
from data import _load_sample
from model_evaluation import load_cached_run, load_validation_predictions
from model_components import Classifier
from model_utils import laplacian_positional_encoding
from IPython.display import display
import pandas as pd

REPORT_DIR = Path.cwd().resolve()
output_dir = REPORT_DIR.parent / "experiments/deterministic_laplacian"
output_dir.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()

## Canonical basis construction

In [2]:
def normalized_laplacian(graph):
    n = graph.num_nodes()
    source, target = graph.edges()
    adjacency = torch.bincount(source.cpu() * n + target.cpu(), minlength=n*n).reshape(n,n).double()
    if not torch.equal(adjacency, adjacency.T):
        raise ValueError("Expected symmetric adjacency with stored multiplicity.")
    degree = adjacency.sum(1)
    inverse = degree.clamp_min(1).rsqrt()
    laplacian = torch.eye(n, dtype=torch.float64) - inverse[:,None]*adjacency*inverse[None,:]
    laplacian[degree == 0, degree == 0] = 0
    return laplacian.numpy()


def canonical_eigenbasis(eigenvalues, eigenvectors, dimensions=8, tolerance=1e-8):
    """Remove sign and repeated-eigenspace rotation ambiguity for a fixed node order."""
    order = np.argsort(eigenvalues, kind="stable")
    values = np.asarray(eigenvalues)[order]
    vectors = np.asarray(eigenvectors, dtype=np.float64)[:,order]
    n = len(values)
    groups = []
    start = 0
    while start < n:
        stop = start + 1
        while stop < n and abs(values[stop] - values[start]) <= tolerance:
            stop += 1
        if values[start] > tolerance:
            group = vectors[:,start:stop]
            projector = group @ group.T
            basis = []
            # Project e_0, e_1, ... in fixed graph-node order. Never pivot on
            # numerical near-ties between norms of interchangeable eigenvectors.
            for anchor in range(n):
                candidate = projector[:,anchor].copy()
                for _ in range(2):  # Reorthogonalize for numerical stability.
                    for previous in basis:
                        candidate -= previous * previous.dot(candidate)
                length = np.linalg.norm(candidate)
                if length <= tolerance:
                    continue
                candidate /= length
                if candidate[anchor] < 0:
                    candidate *= -1
                basis.append(candidate)
                if len(basis) == stop - start:
                    break
            if len(basis) != stop - start:
                raise ValueError("Failed to recover the complete eigenspace rank.")
            groups.append(np.column_stack(basis))
        start = stop
    result = np.zeros((n,dimensions), dtype=np.float64)
    available = np.column_stack(groups)[:,:dimensions] if groups else np.empty((n,0))
    result[:,:available.shape[1]] = available
    return result


def deterministic_laplacian_encoding(graph, dimensions=8, backend="numpy"):
    matrix = normalized_laplacian(graph)
    if backend == "torch":
        values, vectors = torch.linalg.eigh(torch.from_numpy(matrix))
        values, vectors = values.numpy(), vectors.numpy()
    elif backend == "numpy":
        values, vectors = np.linalg.eigh(matrix)
    else:
        values, vectors = linalg.eigh(matrix, driver=backend)
    return canonical_eigenbasis(values, vectors, dimensions)

## Sign flips, eigenspace rotations and numerical backends

In [3]:
# A cube graph has repeated non-zero eigenvalues, including a block cut by k=4.
import dgl
edges = [(i, i ^ (1 << bit)) for i in range(8) for bit in range(3)]
graph = dgl.graph(([a for a,b in edges], [b for a,b in edges]), num_nodes=8)
matrix = normalized_laplacian(graph)
values, vectors = np.linalg.eigh(matrix)
reference = canonical_eigenbasis(values, vectors, dimensions=4)
rng = np.random.RandomState(0)
rotated = vectors.copy()
for value in np.unique(np.round(values, 8)):
    indices = np.flatnonzero(np.abs(values-value) < 1e-8)
    rotation, _ = np.linalg.qr(rng.randn(len(indices),len(indices)))
    rotated[:,indices] = vectors[:,indices] @ rotation
rotation_difference = float(np.max(np.abs(reference - canonical_eigenbasis(values,rotated,4))))
assert rotation_difference < 1e-10
fixture_differences = {backend: float(np.max(np.abs(reference - deterministic_laplacian_encoding(graph,4,backend))))
                       for backend in ["numpy", "torch", "evd", "evr", "ev"]}
assert max(fixture_differences.values()) < 1e-8
print("Rotated/re-signed eigenspaces: max difference", rotation_difference)
print("Backend differences:", fixture_differences)

Rotated/re-signed eigenspaces: max difference 2.220446049250313e-16
Backend differences: {'numpy': 0.0, 'torch': 5.273559366969494e-16, 'evd': 3.608224830031759e-16, 'evr': 5.828670879282072e-16, 'ev': 3.3306690738754696e-16}


## Full validation-set check

Rebuild eight coordinates for every validation face. Compare NumPy with PyTorch and two SciPy eigensolvers. Compare classifier predictions across these canonical bases and with the original cached predictions. Reuse the retained frozen geometry cache; its encoder weights match the Laplacian checkpoint.

In [4]:
load_cached_run("laplacian_frozen")
root = REPORT_DIR / "data/model_runs/laplacian_frozen/epoch_020"
checkpoint = torch.load(root / "checkpoint.pt", map_location="cpu", weights_only=True)
reference = load_validation_predictions(root)
wl_root = REPORT_DIR / "data/model_runs/wl3_frozen/epoch_020"
wl_checkpoint_geometry = np.load(wl_root / "geometry.npy")
baseline = torch.load(REPORT_DIR / "data/model_runs/baseline/epoch_020/checkpoint.pt", map_location="cpu", weights_only=True)
assert all(torch.equal(value, baseline["model_state_dict"]["0." + key])
           for key,value in checkpoint["extra"]["face_cnn_state_dict"].items())
head = Classifier(72,128,16)
head.load_state_dict(checkpoint["model_state_dict"])
head.eval()
backends = ["numpy", "torch", "evd", "evr"]
parts = {backend: [] for backend in backends}
max_difference = {backend: 0.0 for backend in backends}
max_orthogonality_error = 0.0
for sample_index, sample_id in enumerate(reference["sample_ids"]):
    solid = _load_sample(sample_id)
    encodings = {backend: deterministic_laplacian_encoding(solid["graph"],8,backend)
                 for backend in backends}
    canonical = encodings["numpy"]
    active = np.linalg.norm(canonical,axis=0) > 0
    if active.any():
        gram = canonical[:,active].T @ canonical[:,active]
        max_orthogonality_error = max(max_orthogonality_error,float(np.abs(gram-np.eye(active.sum())).max()))
    for backend in backends:
        parts[backend].append(encodings[backend].astype(np.float32))
        max_difference[backend] = max(max_difference[backend],float(np.abs(encodings[backend]-canonical).max()))
    if sample_index % 600 == 0:
        print(f"Checked {sample_index:,}/{len(reference['sample_ids']):,} solids",flush=True)
encodings = {backend: np.concatenate(rows) for backend,rows in parts.items()}
assert all(array.shape == (len(reference["labels"]),8) for array in encodings.values())
rows = []
probabilities_by_backend = {}
with torch.inference_mode():
    for backend,array in encodings.items():
        full = torch.from_numpy(np.concatenate([wl_checkpoint_geometry,array],axis=1))
        probabilities = head(full).softmax(1)
        probabilities_by_backend[backend] = probabilities
        predicted = probabilities.argmax(1)
        rows.append(dict(backend=backend, max_coordinate_difference=max_difference[backend],
                         validation_accuracy=float((predicted==reference["labels"]).double().mean()),
                         changed_from_original=int((predicted!=reference["probabilities"].argmax(1)).sum()),
                         changed_from_canonical_numpy=int((predicted!=probabilities_by_backend["numpy"].argmax(1)).sum())))
comparison = pd.DataFrame(rows).set_index("backend")
display(comparison)
summary = dict(validation_faces=len(reference["labels"]), validation_solids=len(reference["sample_ids"]),
               fixture_rotation_max_difference=rotation_difference,
               fixture_backend_differences=fixture_differences,
               max_orthogonality_error=max_orthogonality_error,
               backend_comparison=rows, elapsed_seconds=time.perf_counter()-started,
               published_caches_unchanged=True)
np.save(output_dir / "canonical_coordinates.npy",encodings["numpy"])
comparison.to_csv(output_dir / "backend_comparison.csv")
(output_dir / "summary.json").write_text(json.dumps(summary,indent=2)+"\n")
print(json.dumps(summary,indent=2))

Checked 0/3,090 solids


/Users/yongjun/miniconda3/envs/gnn_report/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Checked 600/3,090 solids


Checked 1,200/3,090 solids


Checked 1,800/3,090 solids


Checked 2,400/3,090 solids


Checked 3,000/3,090 solids


,max_coordinate_difference,validation_accuracy,changed_from_original,changed_from_canonical_numpy
backend,,,,
numpy,0.000000e+00,0.909615,269,0
torch,1.313859e-12,0.909615,269,0
evd,6.719275e-13,0.909615,269,0
evr,8.462120e-13,0.909615,269,0


{
  "validation_faces": 70255,
  "validation_solids": 3090,
  "fixture_rotation_max_difference": 2.220446049250313e-16,
  "fixture_backend_differences": {
    "numpy": 0.0,
    "torch": 5.273559366969494e-16,
    "evd": 3.608224830031759e-16,
    "evr": 5.828670879282072e-16,
    "ev": 3.3306690738754696e-16
  },
  "max_orthogonality_error": 2.1231564898698855e-15,
  "backend_comparison": [
    {
      "backend": "numpy",
      "max_coordinate_difference": 0.0,
      "validation_accuracy": 0.9096149740232012,
      "changed_from_original": 269,
      "changed_from_canonical_numpy": 0
    },
    {
      "backend": "torch",
      "max_coordinate_difference": 1.3138589407057756e-12,
      "validation_accuracy": 0.9096149740232012,
      "changed_from_original": 269,
      "changed_from_canonical_numpy": 0
    },
    {
      "backend": "evd",
      "max_coordinate_difference": 6.719275301765412e-13,
      "validation_accuracy": 0.9096149740232012,
      "changed_from_original": 269,
      